# Mervis -- Phase 1: Fine-tune Phi-4-mini on Google Colab

Fine-tune **microsoft/Phi-4-mini-instruct** to answer as the two-headed robot
(**Mervin** the gloomy one, **Mervis** the cheerful one) using **QLoRA** so it
fits on a free Colab **T4 (16 GB)**.

**How to run:** `Runtime -> Run all`. The notebook clones this repo to fetch the
dataset automatically. At the end you get a `mervis-merged.zip` containing the
merged model for Phase 2.

> **GPU check:** make sure you picked a GPU runtime --
> `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`.

## 1. Install dependencies

Pinned to a known-compatible set so the notebook 'just runs'.

In [ ]:
%pip install -q \
  "transformers==4.49.0" \
  "trl==0.14.0" \
  "peft==0.14.0" \
  "accelerate==1.3.0" \
  "bitsandbytes==0.45.3" \
  "datasets==3.2.0" \
  "sentencepiece" \
  "tiktoken"

In [ ]:
# Sanity check: confirm a GPU is attached.
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Set Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the dataset

Clone the public repo to pull `mervin_mervis_finetune.csv` (and this notebook's
sibling files) onto the Colab VM. Re-running is safe -- it skips the clone if the
repo is already present.

In [ ]:
import os
from datasets import load_dataset

REPO_DIR = "mervis"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/freeideas/mervis.git

CSV_PATH = os.path.join(REPO_DIR, "mervin_mervis_finetune.csv")
assert os.path.exists(CSV_PATH), f"CSV not found at {CSV_PATH}"

raw = load_dataset("csv", data_files=CSV_PATH, split="train")
print(raw)
print(raw[0])

## 3. Model + tokenizer (4-bit base)

Load Phi-4-mini in 4-bit (QLoRA) via bitsandbytes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.config.use_cache = False
print("Loaded", BASE_MODEL)

## 4. Render each row into the Phi-4 chat template

We let the tokenizer build the `<|user|> ... <|assistant|> ...` string so it
exactly matches what Phi-4-mini expects. The full `response` (both `<Mervin>`
and `<Mervis>` tags) is the assistant turn.

In [ ]:
def to_text(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset[0]["text"])

## 5. LoRA config

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

## 6. Train

~262 examples, 3 epochs. On a T4 this takes roughly 10-20 minutes.

In [ ]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "mervis-lora"

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapters to", ADAPTER_DIR)

## 7. Quick smoke test

Generate from the freshly trained adapters to confirm both personas appear.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is the capital of France?"}],
    tokenize=False, add_generation_prompt=True,
)
out = gen(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"]
print(out[len(prompt):])

## 8. Merge LoRA into the base weights

Reload the base in fp16 (no quantization), apply the adapters, and merge.
Merging into a 4-bit model isn't supported, so we reload fresh.

This cell is **self-contained**: it reloads the adapters from disk
(`mervis-lora/`) and the cached base, so it still works if the kernel was
restarted after training -- the saved adapters are all that's needed.

> **Heads-up on a standard (free) runtime:** the full fp16 model is ~7.6 GB and
> standard Colab gives only ~13 GB RAM, so this step can be **slow** (minutes,
> and the progress bar may look stuck while it's actually working). It does
> finish. For a fast merge, use a **High-RAM** runtime
> (`Runtime -> Change runtime type -> High-RAM`, requires Colab Pro).

In [ ]:
import gc, torch
from peft import PeftModel

BASE_MODEL  = "microsoft/Phi-4-mini-instruct"
ADAPTER_DIR = "mervis-lora"
MERGED_DIR  = "mervis-merged"

# Free anything still pinning the GPU before the fp16 load. The smoke-test
# `gen` pipeline holds the old 4-bit model, so dropping `trainer`/`model`
# alone isn't enough -- drop `gen` too. pop() = no error if already gone.
for _n in ["gen", "trainer", "model", "merged", "base"]:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

# device_map={"": 0} -> whole model on GPU 0, no silent CPU/disk offload.
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()
# 2 GB shards: keeps files manageable and under common upload limits.
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

## 9. Save the merged model to Google Drive

A multi-GB `files.download()` over the browser is flaky (no resume, dies if the
tab hiccups). Saving to Drive is reliable and resumable, survives a dead Colab
session, and you can pull it to your machine later via Drive desktop/web.

Running the mount cell pops a **Google OAuth dialog in the Colab tab** -- approve
it (account chooser -> allow access) and the cell continues.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil
DRIVE = "/content/drive/MyDrive"

# Save the adapter first (irreplaceable -- the merged model is just
# base + adapter), then the full merged model.
for name in ["mervis-lora", "mervis-merged"]:
    src, dst = name, os.path.join(DRIVE, name)
    if os.path.isdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        sz = sum(os.path.getsize(os.path.join(r, f))
                 for r, _, fs in os.walk(dst) for f in fs) / 1e9
        print(f"Saved {src} -> {dst}  ({sz:.2f} GB)")
    else:
        print(f"skip {src} (not present)")

In [ ]:
# --- Alternative: push to the Hugging Face Hub (ideal home for Phase 2, since
# Transformers.js / WebLLM can load a model directly from an HF repo URL). ---
# from huggingface_hub import login
# login()  # paste a write token
# merged.push_to_hub("your-username/mervis-phi4-mini")
# tokenizer.push_to_hub("your-username/mervis-phi4-mini")